# Rigorous Pipeline Experimentation Suite

This notebook implements a rigorous experimentation protocol for the peptide generation pipeline.
It explores the impact of hyperparameters on pipeline performance by modifying exactly one parameter at a time from a stable baseline configuration.

## Experiments Structure

1.  **Framework Setup**: Imports, Data Loading, Model Helpers, Base Configuration.
2.  **Experiment 1: Target Size Sensitivity** (Length 2 to 12).
3.  **Experiment 2: CVAE Hyperparameters** (Latent Dimension, Hidden Dimension, Learning Rate, Epochs).
4.  **Experiment 3: Orchestrator Strategy** (Exploration Rate, Parent Count).
5.  **Analysis**: Automated aggregation and visualization of results.

## Configuration

- **Results Storage**: All metrics are collected in `EXPERIMENT_RESULTS` and saved to CSV.
- **Run Profile**: Set `RUN_PROFILE` to `'fast'` for debugging (fewer repeats) or `'full'` for rigorous testing.
- **Reproducibility**: Seeds are fixed for consistency. 

Run the cells in order. You can stop after any experiment block; results accumulate in the `EXPERIMENT_RESULTS` list.


In [ ]:
import os
import sys
import gc
import math
import json
import time
import random
from pathlib import Path
from typing import Any, Dict, List, Tuple
from itertools import product

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Project Imports
from peptide_pipeline.dataloader.dataloader_json import JSONDataLoader
from peptide_pipeline.generator.cvae_generator import CVAEGenerator
from peptide_pipeline.chemist.agent_v1.chemist_agent import ChemistAgent
from peptide_pipeline.chemist.agent_v1.config_chemist import ChemistConfig, RangeTarget
from peptide_pipeline.biologist.esm_biologist_cos import ESMBiologistCos
from peptide_pipeline.orchestrator.orchestrator import Orchestrator

DATA_PATH = REPO_ROOT / 'database' / 'training_data.json'
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'pipeline_experimentations'
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

RUN_PROFILE = 'full'  # 'fast' or 'full'
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
try:
    torch.random.manual_seed(SEED)
except Exception:
    pass

print(f'Repo root: {REPO_ROOT}')
print(f'Data path exists: {DATA_PATH.exists()}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Run profile: {RUN_PROFILE}')

Repo root: /home/arthur/Documents/M2/pfe/multi-expert-peptide-pipeline
Data path exists: True
CUDA available: True
Run profile: full


In [15]:
from peptide_pipeline.dataloader.dataloader_json import DataLoader as JSONDataLoader
from peptide_pipeline.generator.cvae_generator import CVAEGenerator
from peptide_pipeline.chemist.agent_v1.config_chemist import ChemistConfig, RangeTarget
from peptide_pipeline.chemist.agent_v1.chemist_agent import ChemistAgent
from peptide_pipeline.orchestrator.orchestrator import Orchestrator
from peptide_pipeline.biologist.base import BaseBiologist

TARGET = {
    'dbaasp_id': 'DBAASPS_373',
    'sequence': 'KLFKRWKHLFR',
    'length': 11,
    'ph': None,
    'molecular_weight': 1558.9480000000003,
    'logp': -0.992100000000006,
    'net_charge': 5.0,
    'isoelectric_point': 12.18,
    'hydrophobicity': 1.05,
    'cathionicity': 6,
}

AA = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i for i, aa in enumerate(AA)}
PAD_IDX = 20
VOCAB_SIZE = 21
MAX_LEN = 14

class FallbackBiologist(BaseBiologist):
    def __init__(self, reference_peptide: str):
        self.reference_peptide = reference_peptide

    def score_peptides(self, peptides: List[str]) -> List[float]:
        if not peptides:
            return []
        ref_set = set(self.reference_peptide)
        denom = max(len(ref_set), 1)
        return [len(ref_set.intersection(set(p))) / denom for p in peptides]

    def predict_activity(self, peptides: List[str], context: Any = None) -> List[float]:
        if isinstance(context, str) and context.strip():
            previous = self.reference_peptide
            self.reference_peptide = context
            scores = self.score_peptides(peptides)
            self.reference_peptide = previous
            return scores
        return self.score_peptides(peptides)


def build_biologist(reference_peptide: str, score_temperature: float = 50.0):
    if USE_ESM_BIOLOGIST:
        try:
            from peptide_pipeline.biologist.esm_biologist_global_l2 import ESMBiologistGlobalL2
            return ESMBiologistGlobalL2(
                reference_peptide=reference_peptide,
                batch_size=16,
                score_temperature=score_temperature,
            )
        except Exception as e:
            print(f'ESM biologist unavailable, using fallback biologist: {e}')
    return FallbackBiologist(reference_peptide=reference_peptide)

In [ ]:
# --- Data Loading ---
VOCAB_SIZE = 21  # 20 AA + PAD
PAD_IDX = 20
MAX_LEN = 14
AA_TO_IDX = {aa: i for i, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}

def encode_one_hot_with_pad(sequences: List[str], max_len: int = MAX_LEN) -> torch.Tensor:
    x = torch.zeros(len(sequences), max_len * VOCAB_SIZE, dtype=torch.float32)
    for i, seq in enumerate(sequences):
        for pos in range(max_len):
            x[i, pos * VOCAB_SIZE + PAD_IDX] = 1.0
        for pos, aa in enumerate(seq[:max_len]):
            if aa in AA_TO_IDX:
                x[i, pos * VOCAB_SIZE + PAD_IDX] = 0.0
                x[i, pos * VOCAB_SIZE + AA_TO_IDX[aa]] = 1.0
    return x

def build_condition_tensor(dataframe: pd.DataFrame, condition_dim: int = 32) -> torch.Tensor:
    cond = torch.zeros(len(dataframe), condition_dim, dtype=torch.float32)
    # Map dataframe columns to condition tensor explicitly
    cond[:, 0] = torch.tensor(dataframe['length'].values, dtype=torch.float32)
    cond[:, 1] = torch.tensor(dataframe['molecular_weight'].values, dtype=torch.float32)
    cond[:, 2] = torch.tensor(dataframe['net_charge'].values, dtype=torch.float32)
    cond[:, 3] = torch.tensor(dataframe['isoelectric_point'].values, dtype=torch.float32)
    cond[:, 4] = torch.tensor(dataframe['hydrophobicity'].values, dtype=torch.float32)
    cond[:, 5] = torch.tensor(dataframe['cathionicity'].values, dtype=torch.float32)
    cond[:, 6] = 0.5 # Default hydrophobic moment
    cond[:, 7] = torch.tensor(dataframe['logp'].values, dtype=torch.float32)
    # Remaining dimensions filled with defaults or ignored by CVAE depending on implementation
    return cond

loader = JSONDataLoader()
loader.load_data(
    source=str(DATA_PATH),
    columns=[
        'sequence', 'length', 'ph', 'molecular_weight', 'logp',
        'net_charge', 'isoelectric_point', 'hydrophobicity', 'cathionicity'
    ],
    fillna_defaults={
        'length': 10,
        'ph': 7.0,
        'molecular_weight': 1500.0,
        'logp': 0.0,
        'net_charge': 0.0,
        'isoelectric_point': 7.0,
        'hydrophobicity': 0.0,
        'cathionicity': 0.0,
    },
    normalize_sequence=True,
    sequence_column='sequence',
    keep_standard_amino_acids_only=True,
)

df = loader.get_data().copy()
sequences = df['sequence'].tolist()
lengths_t = torch.tensor(df['length'].astype(int).values, dtype=torch.long)
x_t = encode_one_hot_with_pad(sequences, max_len=MAX_LEN)
conditions_t = build_condition_tensor(df, condition_dim=32)

print(f'Dataset rows: {len(df)}')
print(f'x shape: {tuple(x_t.shape)}')
print(f'conditions shape: {tuple(conditions_t.shape)}')
display(df.head(3))

Dataset rows: 4410
x shape: (4410, 294)
conditions shape: (4410, 32)


,sequence,length,ph,molecular_weight,logp,net_charge,isoelectric_point,hydrophobicity,cathionicity
0,KVVVKWVVKVVK,12,7.0,1648.291,5.6026,5.0,14.0,-1.07,4
1,LFIFFF,6,7.0,832.059,3.2860,1.0,14.0,-3.25,0
2,KAAAKWAAKAAK,12,7.0,1451.913,1.1499,5.0,14.0,0.33,4


In [ ]:
def build_chemist_from_target(
    target: Dict[str, Any],
    width_scale: float = 1.0,
    length_min: float = 8.0,
    length_max: float = 14.0,
    length_weight: float = 1.0,
) -> ChemistAgent:
    ph = 7.0 if target['ph'] is None else float(target['ph'])
    target_length = float(target['length'])
    min_len = min(length_min, target_length)
    max_len = max(length_max, target_length)

    target_mw = float(target['molecular_weight'])
    mw_span = max(250.0, 0.35 * target_mw)
    mw_min = max(200.0, target_mw - mw_span)
    mw_max = target_mw + mw_span

    target_charge = float(target['net_charge'])
    charge_span = max(1.5, 0.5 * abs(target_charge))
    charge_min = max(0.0, target_charge - charge_span)
    charge_max = target_charge + charge_span

    target_ip = float(target['isoelectric_point'])
    ip_span = 2.5 * width_scale
    ip_min = max(3.0, target_ip - ip_span)
    ip_max = min(14.0, target_ip + ip_span)

    target_hydro = float(target['hydrophobicity'])
    hydro_span = 1.5 * width_scale
    hydro_min = target_hydro - hydro_span
    hydro_max = target_hydro + hydro_span

    target_logp = float(target['logp'])
    logp_span = 2.0 * width_scale
    logp_min = target_logp - logp_span
    logp_max = target_logp + logp_span

    return ChemistAgent(
        config=ChemistConfig(
            ph=ph,
            length=RangeTarget(min=min_len, max=max_len, target=target_length, weight=length_weight),
            molecular_weight=RangeTarget(min=mw_min, max=mw_max, target=target_mw, weight=1.0),
            logp=RangeTarget(min=logp_min, max=logp_max, target=target_logp, weight=1.0),
            net_charge=RangeTarget(min=charge_min, max=charge_max, target=target_charge, weight=1.0),
            isoelectric_point=RangeTarget(min=ip_min, max=ip_max, target=target_ip, weight=1.0),
            hydrophobicity=RangeTarget(min=hydro_min, max=hydro_max, target=target_hydro, weight=1.0),
        )
    )


def model_cache_path(latent_dim: int, hidden_dim: int, epochs: int, lr: float) -> Path:
    lr_text = str(lr).replace('.', 'p')
    return EXPERIMENT_DIR / f'cvae_lat{latent_dim}_hid{hidden_dim}_ep{epochs}_lr{lr_text}.pth'


def get_or_train_cvae(latent_dim: int, hidden_dim: int, epochs: int, lr: float, kl_anneal_epochs: int, batch_size: int = 64) -> CVAEGenerator:
    model = CVAEGenerator(max_len=MAX_LEN, latent_dim=latent_dim, hidden_dim=hidden_dim, condition_dim=32)
    path = model_cache_path(latent_dim, hidden_dim, epochs, lr)

    if path.exists():
        model.load_model(str(path))
        return model

    device = model.device
    x_local = x.to(device)
    c_local = conditions.to(device)
    l_local = lengths.to(device)

    model.train_model(
        data=x_local,
        conditions=c_local,
        lengths=l_local,
        epochs=epochs,
        batch_size=batch_size,
        lr=lr,
        kl_anneal_epochs=kl_anneal_epochs,
    )
    model.save_model(str(path))
    return model

In [ ]:
# --- Experiment Framework ---

EXPERIMENT_RESULTS = []

# Base Configuration
BASE_CONFIG = {
    # CVAE
    'cvae_latent_dim': 32,
    'cvae_hidden_dim': 128,
    'cvae_epochs': 60, # Reduced for speed in tests, increase for final
    'cvae_lr': 0.001,
    
    # Orchestrator
    'orch_iterations': 10,
    'orch_peptides_per_iter': 100,
    'orch_top_k': 10,
    'orch_exploration_rate': 0.2,
    'orch_parent_count': 4,
    
    # Target (Chemist)
    'target_length': 6,
    'target_mw': 800.0,
    'target_charge': 0.0,
    'target_ph': 7.0,
    'target_ip': 7.0,
    'target_hydro': 0.0,
    'target_logp': 0.0,
}

def run_single_experiment(
    config_overrides: Dict[str, Any],
    run_id: int,
    experiment_name: str,
    biologist_ref: ESMBiologistCos
) -> Dict[str, Any]:
    
    # Merge base config with overrides
    cfg = BASE_CONFIG.copy()
    cfg.update(config_overrides)
    
    # 1. Get/Train Model
    # Note: Training relies on global x_t, conditions_t, lengths_t loaded earlier
    cvae = get_or_train_cvae(
        x_train=x_t, 
        c_train=conditions_t, 
        l_train=lengths_t,
        latent_dim=cfg['cvae_latent_dim'],
        hidden_dim=cfg['cvae_hidden_dim'],
        epochs=cfg['cvae_epochs'],
        lr=cfg['cvae_lr']
    )
    
    # 2. Setup Chemist
    target_def = {
        'length': cfg['target_length'],
        'molecular_weight': cfg['target_mw'],
        'net_charge': cfg['target_charge'],
        'ph': cfg['target_ph'],
        'isoelectric_point': cfg['target_ip'],
        'hydrophobicity': cfg['target_hydro'],
        'logp': cfg['target_logp']
    }
    chemist = build_chemist_from_target(target_def)
    
    # 3. Setup Orchestrator
    orchestrator = Orchestrator(
        generator=cvae,
        chemist=chemist,
        biologist=biologist_ref
    )
    
    # 4. Run Pipeline
    # Infer final target for orchestrator from chemist config inside orchestrator, 
    # but we can pass explicit dictionary if needed.
    # The orchestrator uses chemist-inferred constraints by default.
    start_time = time.time()
    results = orchestrator.run(
        nb_iterations=cfg['orch_iterations'],
        nb_peptides=cfg['orch_peptides_per_iter'],
        top_k=cfg['orch_top_k'],
        exploration_rate=cfg['orch_exploration_rate'],
        random_parent_count=cfg['orch_parent_count']
    )
    duration = time.time() - start_time
    
    # 5. Summarize
    # Create a reference sequence (e.g. poly-A of target length) for novelty ref or use something else
    # Here we use 'A'*length as placeholder reference
    ref_seq = "A" * int(cfg['target_length'])
    
    summary = summarize_topk(results, target_def, reference_seq=ref_seq)
    
    # Add config & metadata
    row = {
        'experiment': experiment_name,
        'run_id': run_id,
        'duration': duration,
        **cfg,       # Flatten config cols
        **summary    # Flatten metric cols
    }
    return row

# Initialize Biologist once to reuse model loading
print("Initializing Biologist...")
biologist_instance = ESMBiologistCos() 
print("Biologist Ready.")

In [ ]:
# --- Analysis & Visualization ---

# Convert results to DataFrame
full_df = pd.DataFrame(EXPERIMENT_RESULTS)

if not full_df.empty:
    # Save results
    save_path = EXPERIMENT_DIR / f'rigorous_experiment_results_{int(time.time())}.csv'
    full_df.to_csv(save_path, index=False)
    print(f"Results saved to {save_path}")

    # Visualization Helpers
    def plot_metric_vs_param(df, param_col, metric_col, title_suffix=""):
        plt.figure(figsize=(8, 5))
        # Aggregate mean and CI for cleaner plots if seaborn errorbar/ci is weird with few points
        sns.lineplot(data=df, x=param_col, y=metric_col, marker='o', errorbar='sd')
        plt.title(f'{metric_col} vs {param_col} {title_suffix}')
        plt.grid(True, alpha=0.3)
        plt.xlabel(param_col)
        plt.ylabel(metric_col)
        plt.show()

    print("=== Analysis Report ===")

    # 1. Target Size Analysis
    subset = full_df[full_df['experiment'] == 'target_size']
    if not subset.empty:
        print("\n--- Target Size Analysis ---")
        plot_metric_vs_param(subset, 'target_length', 'top1_score', "(Target Size Exp)")
        plot_metric_vs_param(subset, 'target_length', 'validity_rate', "(Target Size Exp)")
        plot_metric_vs_param(subset, 'target_length', 'unique_ratio', "(Target Size Exp)")

    # 2. CVAE Params Analysis
    cvae_params = [
        ('cvae_latent_dim', 'cvae_latent_dim'),
        ('cvae_hidden_dim', 'cvae_hidden_dim'),
        ('cvae_lr', 'cvae_lr'),
        ('cvae_epochs', 'cvae_epochs')
    ]
    
    for exp_name, param_col in cvae_params:
        subset = full_df[full_df['experiment'] == exp_name]
        if not subset.empty:
            print(f"\n--- {exp_name} Analysis ---")
            plot_metric_vs_param(subset, param_col, 'top1_score')
            plot_metric_vs_param(subset, param_col, 'validity_rate')

    # 3. Orch Params Analysis
    orch_params = [
        ('orch_exploration', 'orch_exploration_rate'),
        ('orch_parent_count', 'orch_parent_count')
    ]

    for exp_name, param_col in orch_params:
        subset = full_df[full_df['experiment'] == exp_name]
        if not subset.empty:
            print(f"\n--- {exp_name} Analysis ---")
            plot_metric_vs_param(subset, param_col, 'top1_score')
            plot_metric_vs_param(subset, param_col, 'unique_ratio')

else:
    print("No results found. Please run the experiment cells above.")

In [ ]:
# --- Experiment 3: Orchestrator Hyperparameters ---

print("Starting Experiment 3: Orchestrator...")
results_exp3 = []

# 3.1 Exploration Rate
rates = [0.0, 0.1, 0.3, 0.5, 0.8]
for val in rates:
    if val == BASE_CONFIG['orch_exploration_rate']: continue
    print(f"  > Testing Exploration Rate: {val}")
    for i in range(N_REPEATS):
        row = run_single_experiment(
            config_overrides={'orch_exploration_rate': val},
            run_id=i,
            experiment_name="orch_exploration",
            biologist_ref=biologist_instance
        )
        results_exp3.append(row)

# 3.2 Parent Count
counts = [1, 2, 4, 8, 16]
for val in counts:
    if val == BASE_CONFIG['orch_parent_count']: continue
    print(f"  > Testing Parent Count: {val}")
    for i in range(N_REPEATS):
        row = run_single_experiment(
            config_overrides={'orch_parent_count': val},
            run_id=i,
            experiment_name="orch_parent_count",
            biologist_ref=biologist_instance
        )
        results_exp3.append(row)

df_exp3 = pd.DataFrame(results_exp3)
EXPERIMENT_RESULTS.extend(results_exp3)
print("Experiment 3 Done.")

In [ ]:
# --- Experiment 2: CVAE Hyperparameters ---
# "Each configurations should try to modify ONLY and EXACTLY one hyper parameters."
# Base Config is used as reference.

print("Starting Experiment 2: CVAE Hyperparams...")
results_exp2 = []

# 2.1 Latent Dimension
latent_dims = [16, 32, 64, 128]
for val in latent_dims:
    if val == BASE_CONFIG['cvae_latent_dim']: continue # Skip base
    print(f"  > Testing Latent Dim: {val}")
    for i in range(N_REPEATS):
        row = run_single_experiment(
            config_overrides={'cvae_latent_dim': val},
            run_id=i,
            experiment_name="cvae_latent_dim",
            biologist_ref=biologist_instance
        )
        results_exp2.append(row)

# 2.2 Hidden Dimension
hidden_dims = [64, 128, 256, 512]
for val in hidden_dims:
    if val == BASE_CONFIG['cvae_hidden_dim']: continue
    print(f"  > Testing Hidden Dim: {val}")
    for i in range(N_REPEATS):
        row = run_single_experiment(
            config_overrides={'cvae_hidden_dim': val},
            run_id=i,
            experiment_name="cvae_hidden_dim",
            biologist_ref=biologist_instance
        )
        results_exp2.append(row)

# 2.3 Learning Rate
lrs = [0.0001, 0.001, 0.005, 0.01]
for val in lrs:
    if val == BASE_CONFIG['cvae_lr']: continue
    print(f"  > Testing LR: {val}")
    for i in range(N_REPEATS):
        row = run_single_experiment(
            config_overrides={'cvae_lr': val},
            run_id=i,
            experiment_name="cvae_lr",
            biologist_ref=biologist_instance
        )
        results_exp2.append(row)

# 2.4 Epochs
epochs_list = [40, 60, 100, 200]
for val in epochs_list:
    if val == BASE_CONFIG['cvae_epochs']: continue
    print(f"  > Testing Epochs: {val}")
    for i in range(N_REPEATS):
        row = run_single_experiment(
            config_overrides={'cvae_epochs': val},
            run_id=i,
            experiment_name="cvae_epochs",
            biologist_ref=biologist_instance
        )
        results_exp2.append(row)

df_exp2 = pd.DataFrame(results_exp2)
EXPERIMENT_RESULTS.extend(results_exp2)
print("Experiment 2 Done.")

In [ ]:
# --- Experiment 1: Target Size Influence ---
# "try to change the target size from 2 to 12"

print("Starting Experiment 1: Target Size...")

# Range 2 to 12
target_sizes = list(range(2, 13)) 
# Number of repeats per configuration
N_REPEATS = 3 if RUN_PROFILE == 'fast' else 10

results_exp1 = []

for size in target_sizes:
    print(f"  > Testing Target Size: {size}")
    
    # We update ONLY the target length (and implicitly MW since it scales with length, but we should probably keep other targets static or scale them?
    # The requirement is "modify ONLY and EXACTLY one hyper parameters".
    # Length is the parameter. But MW usually correlates.
    # Theoretically if we change length, we should update MW target or let chemist ignore it?
    # Our `build_chemist` function infers MW range from target_mw. 
    # If we keep target_mw=800 (default) for length=2 (too heavy) or length=12 (maybe too light?), it might conflict.
    # However, strictly following "modify one thing", we keep target_mw constant.
    # BUT, 'size' changes physical reality.
    # Let's adjust target_mw to be realistic: ~110 Da per AA.
    # Using dynamic MW based on size might violate "one param" rule if strict, but 'Size' implies corresponding mass.
    # Let's update MW to be consistent with size: 110 * size.
    
    current_overrides = {
        'target_length': size,
        'target_mw': 110.0 * size # Adjusting this to be chemically consistent
    }
    
    for i in range(N_REPEATS):
        row = run_single_experiment(
            config_overrides=current_overrides,
            run_id=i,
            experiment_name="target_size",
            biologist_ref=biologist_instance
        )
        results_exp1.append(row)

df_exp1 = pd.DataFrame(results_exp1)
EXPERIMENT_RESULTS.extend(results_exp1)
print("Experiment 1 Done.")
display(df_exp1.groupby('target_length')[['top1_score', 'validity_rate', 'unique_ratio', 'diversity', 'novelty_vs_reference']].mean(numeric_only=True))